In [27]:
import os
import sys
import pickle
import numpy as np
from numba import njit
import itertools as itt
import aerosandbox as asb

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from Aircraft.Planform import Planform
from Aircraft.Fixed import Fixed
from Drag.Fuselage import Fuselage
from Drag.Bay import Bay
from Drag.LandingGear import LandingGear
from Aircraft.Aircraft import Aircraft
from global_parameters import Assumptions
from Requirements.FuelReq import FuelReq
from Requirements.LGReq import LGReq
from Requirements.MassReq import MassReq
from Requirements.MDReq import MDReq
from Requirements.EmpennageReq import EmpennageReq
from Requirements.Requirement import Requirement
from EmpennageSizing.TailFinder import TailFinder
from EmpennageSizing.CanardFinder import CanardFinder
from structural_analysis.Material import Material

# Loading the pre-computed planforms and the fuselage

In [28]:
with open("pickles/planform_pickle_official.pcl", "r+b") as f:
    plaforms_recovered:list[tuple[Planform, str, bool]] = pickle.load(f)

assumptions = Assumptions()

In [29]:
# # --- Import Onshape pull utilities ---
# sys.path.append(os.path.abspath(os.getcwd()))
# from onshape_pull import (
#     fetch_variable_studio, fetch_measurement_features,
#     evaluate_measurements, load_cached_masses, fetch_mass_properties,
#     compute_cg_scenarios, lookup_var, lookup_meas,
#     UPDATE_MASSES,
# )

# # --- Pull data from Onshape ---
# variables = fetch_variable_studio()
# meas_names = fetch_measurement_features()
# measurements = evaluate_measurements(meas_names)
# components = load_cached_masses() if not UPDATE_MASSES else fetch_mass_properties()
# cg_data = compute_cg_scenarios(components)

# # --- Z offset (axle datum) ---
# Front_Landing_Gear_Hinge_Z = lookup_meas(measurements, "Front_Landing_Gear_Hinge_Z")
# Front_Strut_Height, _, _ = lookup_var(variables, "Front_Strut_Height")
# Front_Gear_Extension_Max = lookup_meas(measurements, "Front_Gear_Extension_Max")
# Front_Gear_Extension_Min = lookup_meas(measurements, "Front_Gear_Extension_Min")
# z_offset = (Front_Landing_Gear_Hinge_Z + Front_Strut_Height
#             + (Front_Gear_Extension_Max - Front_Gear_Extension_Min))

# # --- Build drag components ---
# engine_bay = Bay(
#     surface_wetted=83744.32631 / 1e6,  # mm² → m² (hardcoded, not in Onshape)
#     length=0.172,                       # 172 mm (hardcoded, not in Onshape)
#     diameter=lookup_var(variables, "engine_diameter")[0],
# )

# Front_Gear_Unexposed = lookup_meas(measurements, "Front_Gear_Unexposed")
# nose_gear = LandingGear(
#     wheel_width=0.025,
#     exposed_height=Front_Strut_Height - Front_Gear_Unexposed,
#     wheel_diameter=lookup_var(variables, "Wheel_Diameter")[0],
#     strut_width=lookup_var(variables, "Front_Strut_Diameter")[0],
# )

# Rear_Strut_Height, _, _ = lookup_var(variables, "Rear_Strut_Height")
# Rear_Strut_height_2, _, _ = lookup_var(variables, "Rear_Strut_height_2")
# main_gear = LandingGear(
#     wheel_width=0.025,
#     exposed_height=Rear_Strut_Height + Rear_Strut_height_2,
#     wheel_diameter=lookup_var(variables, "Wheel_Diameter")[0],
#     strut_width=lookup_var(variables, "Rear_Strut_Diameter")[0],
# )

# fuselage = Fuselage(
#     surface_wetted=lookup_meas(measurements, "Wetted_Area"),
#     length_total=lookup_var(variables, "FuselageLength")[0],
#     diameter_max=lookup_var(variables, "FuselageHeight")[0],
#     upsweep=0.0,
#     base_area=lookup_meas(measurements, "Base_Area"),
# )

# # --- X-position helpers ---
# WingPortDistance, _, _ = lookup_var(variables, "FuselageLength")
# WingPortDistance = (WingPortDistance / 2) - 0.025
# WingPortWidth, _, _ = lookup_var(variables, "WingPortWidth")
# CanardPortXLoc, _, _ = lookup_var(variables, "CanardPortXLoc")
# CanardPortWidth, _, _ = lookup_var(variables, "CanardPortWidth")

# print(cg_data["x_cg_min"])
# print(cg_data["x_cg_max"])


# # --- Fixed parameters ---
# fixed = Fixed(
#     mass=cg_data["mass"],
#     fuel_mass=cg_data["fuel_mass"],
#     x_cg_min=cg_data["x_cg_min"],
#     x_cg_max=cg_data["x_cg_max"],
#     x_tail_cone=lookup_meas(measurements, "Tailcone_X"),
#     z_cg=cg_data["z_cg_full"] + z_offset,
#     z_tail_cone=-lookup_meas(measurements, "Z_TailCone") + z_offset,
#     z_wing=lookup_meas(measurements, "Z_wing_LE_Abs") + z_offset,
#     x_LE_canard=CanardPortXLoc + CanardPortWidth / 2,
#     x_LE_wing=WingPortDistance + 0.115 - int(WingPortWidth) / 2,
#     x_LE_tail=lookup_meas(measurements, "X_LE_Tail"),
#     x_nose_gear=lookup_meas(measurements, "x_nose_gear"),
#     x_main_gear=lookup_meas(measurements, "x_main_gear"),
#     y_main_gear=0.419,
#     fuselage=fuselage,
#     nose_gear=nose_gear,
#     main_gear=main_gear,
#     engine_bay=engine_bay,
# )

In [30]:
# with open("pickles/fixed_pickle.pcl", "wb") as f:
#     pickle.dump(fixed, f)

In [31]:
with open("pickles/fixed_pickle.pcl", "rb") as f:
    fixed:Fixed = pickle.load(f)

In [32]:
print(fixed.x_cg_max, fixed.x_cg_min, fixed.x_LE_wing, fixed.z_cg, fixed.z_tail_cone)
fixed.x_LE_wing = 1.255
delta_z = 0.01
fixed.z_tail_cone += delta_z
fixed.z_cg += delta_z
print(fixed.z_cg, fixed.z_tail_cone)
fixed.fuel_mass = 13.54
fixed.x_cg_min = 1.336 #m
fixed.x_cg_max = 1.403 #m
fixed.mass = 42.2 #kg 

1.3589152301106575 1.271545179962353 1.4400000000000002 0.17957635338517247 0.11000000000000004
0.18957635338517248 0.12000000000000004


In [33]:
for component in fixed.drag_components(False):
    component.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
    component.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
    component.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
for component in fixed.drag_components(True):
    component.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)

# Creating full Aircraft objects

In [34]:
material_skin = Material(assumptions.cfrp_density, elastic_modulus=assumptions.cfrp_Young_modulus, 
                         poisson_ratio=assumptions.cfrp_poisson, shear_modulus=assumptions.cfrp_Young_modulus / 2 / (1 + assumptions.cfrp_poisson),
                         yield_strength=assumptions.cfrp_yield_strength, fracture_strength=assumptions.cfrp_yield_strength)

In [35]:
aircraft:list[Aircraft] = list()

for i, planform_recovered in enumerate(plaforms_recovered):
    main_wing = planform_recovered[0]
    planform_type = planform_recovered[1]

    ef = TailFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_h=max(4., main_wing.aspect_ratio/2)) if (planform_type == "tail") else CanardFinder(fixed, material=material_skin, core_density=assumptions.foam_denisty, thicknesses=assumptions.allowable_thicknesses, safety_factor=assumptions.structural_safety_factor, AR_c=max(5., main_wing.aspect_ratio/2))
    
    emp = ef.find_planforms(main_wing, print_=i==28)

    for e in emp:
        e.add_cache_entry("go_around", assumptions.airspeed_approach/asb.Atmosphere(assumptions.altitude_go_round).speed_of_sound(), assumptions.altitude_go_round)
        e.add_cache_entry("mach_max", assumptions.mach_max, assumptions.altitude_mach_max)
        e.add_cache_entry("cruise", assumptions.mach_cruise, assumptions.altitude_cruise)
        e.add_cache_entry("takeoff", assumptions.airspeed_approach/asb.Atmosphere().speed_of_sound(), 0.)
        e.mass_cache = 1
        e.x_cg_cache = .1

    aircraft_planforms = [main_wing] + emp #TODO add the empenage
    aircraft.append(Aircraft(
        fixed=fixed, #TODO: add the fuselage from CAD
        planforms=aircraft_planforms 
    ))

Stresses 24013280.215892255, 73961606.64878033, 0.0004
Stresses 16093427.290091699, 51771871.64919781, 0.0005959183673469389
Stresses 72192385.51451235, 800706851.155794, 0.0004
Stresses 48432826.73711061, 616412768.1093066, 0.0005959183673469389
Stresses 36430575.395949006, 533358706.0316137, 0.0007918367346938775
Stresses 29189547.727314267, 492142395.11309445, 0.0009877551020408164
Stresses 24345549.90760687, 472291781.09789777, 0.0011836734693877551
Stresses 20877362.11953238, 464272763.1446708, 0.0013795918367346938
Stresses 18271728.807248417, 462270959.6355071, 0.0015755102040816327
Stresses 16242456.780630963, 461744002.95375997, 0.0017714285714285716
Stresses 14617355.116161391, 458612246.69107413, 0.0019673469387755105
Stresses 13286611.489029665, 449323332.137764, 0.002163265306122449
Stresses 12176891.024881778, 431464395.3700606, 0.002359183673469388
Stresses 11237351.334788527, 404467816.2442257, 0.002555102040816327
Stresses 10431633.32162249, 369826645.6045014, 0.002751

In [36]:
s_ratios = [ac.planforms[1].wing_area / ac.planforms[0].wing_area for ac in aircraft]
print(s_ratios)
print(min(s_ratios), np.average(s_ratios), max(s_ratios))
ac_bad:Aircraft = aircraft[np.argmax(s_ratios)],
print(np.argmax(s_ratios))
ac_bad= ac_bad[0]
print(ac_bad.planforms[0].sweep_quarter_rad, ac_bad.planforms[0].aspect_ratio, ac_bad.planforms[0].cm_quarter_chord, ac_bad.planforms[0].thickness_to_chord)

[0.10252304302328744, 0.01977241382635263, 0.10252304302328744, 0.08005553599054957, 0.1651037760365564, 0.16605485879124363, 0.0727663330759258, 0.07009697443539273, 0.1025839038425511, 0.019688762163821327, 0.1025839038425511, 0.08014031929868116, 0.16502569457400879, 0.16597122683416382, 0.07268593664382873, 0.07001613034036279, 0.10268936143993421, 0.01954380776872843, 0.10268936143993421, 0.08028724264739917, 0.16489038371701203, 0.16582630631375114, 0.07254661165331647, 0.06987604288373825, 0.12639353199518447, 0.025865831512592637, 0.1270995352826047, 0.09926053469834209, 0.17382719552421128, 0.18026483803524176, 0.11142579952143959, 0.11257256825731535, 0.1264804370807562, 0.025982994533433917, 0.1271907160308245, 0.09938942370349745, 0.17372223962181035, 0.1801485862356091, 0.11130940695244931, 0.11244849740763477, 0.1266223561264889, 0.026186025088283316, 0.12735015354735263, 0.0996148125542623, 0.17354036375860346, 0.17994715283292445, 0.11110586111244088, 0.1122315495245018

# Checking if reuirements are met

In [37]:
for ac in aircraft:
    ac.fixed = fixed

In [38]:
requirements:list[Requirement] = [
    MassReq(50.),
    MDReq(),
    FuelReq(),
    LGReq(),
    EmpennageReq(),
]

requirement_labels = [
    "MTOM",
    "Matching Diagram",
    "Fuel",
    "Landing Gear",
    "Empennage Requirement"
]

In [39]:
for ac in aircraft:
    failed_reqs = list()
    for requirement, label in zip(requirements, requirement_labels):
        # if type(requirement) == LGReq:
        #         print("f")
        if not requirement.assess(ac):
            failed_reqs.append(label)

    if len(failed_reqs):
        print(f"ac mass: {ac.total_mass()}, {ac.planforms[0].oswald}")
        print(f"MainWing: AR={ac.planforms[0].aspect_ratio}, tc={ac.planforms[0].thickness_to_chord}, sweep={np.rad2deg(ac.planforms[0].sweep_quarter_rad)} deg, cmac={ac.planforms[0].cm_quarter_chord}")
        print(f"Failed: {failed_reqs}")
        print()